# CAS Exam 5: Bornhuetter-Ferguson, Benktander Methods, and Cape Cod Methods

Dataset used:
- `chainladder-python/chainladder/utils/data/friedland_us_industry_auto.csv`

Learning goals:
- Build BF and Benktander estimates using `chainladder.Triangle` objects.
- Connect method outputs back to the formula-sheet interpretation.
- Show Benktander convergence to Chain Ladder ultimate as iterations increase.


## Formula-Sheet Summary

BF method blends development and expected claims:
- Ultimate = Actual + Expected x % Unreported
- Equivalent credibility view: Ultimate = Development Ultimate x (1/CDF) + Expected x (1 - 1/CDF)

Benktander method:
- Iterative credibility-weighted extension of BF.
- As iterations increase, Benktander approaches development (Chain Ladder) ultimate.




In [10]:
from __future__ import annotations

from pathlib import Path
import sys

import chainladder as cl
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import build_exposure_triangle, triangle_to_frame

DATA_PATH = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data' / 'friedland_us_industry_auto.csv'
raw = pd.read_csv(DATA_PATH)

triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Paid Claims', 'Reported Claims'],
    cumulative=True,
)
reported_triangle = triangle['Reported Claims']

{'triangle_shape': triangle.shape, 'valuation_date': str(triangle.valuation_date), 'columns': list(triangle.columns)}, reported_triangle


({'triangle_shape': (1, 2, 10, 10),
  'valuation_date': '2007-12-31 23:59:59.999999999',
  'columns': ['Paid Claims', 'Reported Claims']},
              12          24          36          48          60          72          84          96          108         120
 1998  37017487.0  43169009.0  45568919.0  46784558.0  47337318.0  47533264.0  47634419.0  47689655.0  47724678.0  47742304.0
 1999  38954484.0  46045718.0  48882924.0  50219672.0  50729292.0  50926779.0  51069285.0  51163540.0  51185767.0         NaN
 2000  41155776.0  49371478.0  52358476.0  53780322.0  54303086.0  54582950.0  54742188.0  54837929.0         NaN         NaN
 2001  42394069.0  50584112.0  53704296.0  55150118.0  55895583.0  56156727.0  56299562.0         NaN         NaN         NaN
 2002  44755243.0  52971643.0  56102312.0  57703851.0  58363564.0  58592712.0         NaN         NaN         NaN         NaN
 2003  45163102.0  52497731.0  55468551.0  57015411.0  57565344.0         NaN         NaN         NaN    

In [11]:
# Development pattern (reported basis)
selected_dev = cl.Development(average='volume', n_periods=-1).fit_transform(reported_triangle)
selected_ldf = selected_dev.ldf_.to_frame()
selected_cdf = selected_dev.cdf_.to_frame()

reported_long = triangle_to_frame(reported_triangle, origin_as_datetime=False).reset_index()
reported_matrix = reported_long.pivot(index='origin', columns='development', values='Reported Claims').sort_index().sort_index(axis=1)
latest_age = reported_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
latest_reported = selected_dev.latest_diagonal.to_frame().iloc[:, 0]

selected_ldf, selected_cdf, latest_age.to_frame(name='LatestAge')


(          12-24     24-36     36-48     48-60     60-72     72-84     84-96  \
 (All)  1.175117  1.058233  1.027177  1.011041  1.004364  1.002609  1.001598   
 
          96-108   108-120  
 (All)  1.000579  1.000369  ,
         12-Ult   24-Ult    36-Ult    48-Ult   60-Ult    72-Ult    84-Ult  \
 (All)  1.30378  1.10949  1.048436  1.020696  1.00955  1.005164  1.002548   
 
          96-Ult   108-Ult  
 (All)  1.000949  1.000369  ,
         LatestAge
 origin           
 1998          120
 1999          108
 2000           96
 2001           84
 2002           72
 2003           60
 2004           48
 2005           36
 2006           24
 2007           12)

## Set the Expected Claims Prior

This dataset does not include premium, so we create an explicit prior input for BF/Benktander demonstration:
- Assume a prior ELR by AY to infer an earned-premium proxy from latest reported.
- Select one portfolio-level apriori ELR used by BF and Benktander estimators.

This is the central actuarial judgment in BF/Benktander.


In [12]:
ay_year = latest_reported.index.year
prior_elr_by_ay = pd.Series(
    [0.720, 0.725, 0.730, 0.735, 0.740, 0.745, 0.750, 0.755, 0.760, 0.765],
    index=ay_year,
    dtype=float,
)

earned_premium_proxy = latest_reported / prior_elr_by_ay.values
selected_apriori = 0.750
expected_ultimate_prior = earned_premium_proxy * selected_apriori

exposure_series = pd.Series(earned_premium_proxy.values, index=ay_year, dtype=float)
sample_weight = build_exposure_triangle(exposure_series, selected_dev)

prior_table = pd.DataFrame(
    {
        'LatestReported': latest_reported.values,
        'PriorELR_ByAY': prior_elr_by_ay.values,
        'EarnedPremiumProxy': earned_premium_proxy.values,
        'SelectedAprioriELR': selected_apriori,
        'ExpectedUltimatePrior': expected_ultimate_prior.values,
    },
    index=ay_year,
)
prior_table.index.name = 'AccidentYear'
prior_table


,LatestReported,PriorELR_ByAY,EarnedPremiumProxy,SelectedAprioriELR,ExpectedUltimatePrior
AccidentYear,,,,,
1998,47742304.0,0.720,6.630876e+07,0.75,4.973157e+07
1999,51185767.0,0.725,7.060106e+07,0.75,5.295079e+07
2000,54837929.0,0.730,7.512045e+07,0.75,5.634034e+07
2001,56299562.0,0.735,7.659804e+07,0.75,5.744853e+07
2002,58592712.0,0.740,7.917934e+07,0.75,5.938451e+07
2003,57565344.0,0.745,7.726892e+07,0.75,5.795169e+07
2004,56976657.0,0.750,7.596888e+07,0.75,5.697666e+07
2005,56786410.0,0.755,7.521379e+07,0.75,5.641034e+07
2006,54641339.0,0.760,7.189650e+07,0.75,5.392237e+07


## BF Estimate: Formula Check and Estimator Output

Reported-basis BF formula used here:
- Ultimate = Latest Reported + Expected Ultimate Prior x (1 - 1/CDF)

Below compares a manual formula implementation to `chainladder.BornhuetterFerguson`.


In [13]:
bf_model = cl.BornhuetterFerguson(apriori=selected_apriori).fit(selected_dev, sample_weight=sample_weight)
bf_ultimate_model = bf_model.ultimate_.to_frame().iloc[:, 0]

cdf_map = selected_dev.cdf_.to_frame().iloc[0]
selected_cdf_by_ay = latest_age.map(lambda age: cdf_map.get(f'{int(age)}-Ult', 1.0))
bf_ultimate_manual = latest_reported + expected_ultimate_prior.values * (1.0 - 1.0 / selected_cdf_by_ay.values)
bf_ultimate_manual = pd.Series(bf_ultimate_manual, index=latest_reported.index)

bf_check = pd.DataFrame(
    {
        'LatestReported': latest_reported,
        'SelectedCDF': selected_cdf_by_ay.values,
        'ExpectedUltimatePrior': expected_ultimate_prior.values,
        'BF_Ultimate_Manual': bf_ultimate_manual.values,
        'BF_Ultimate_Model': bf_ultimate_model.values,
    },
    index=ay_year,
)
bf_check['AbsDiff_Manual_vs_Model'] = (bf_check['BF_Ultimate_Manual'] - bf_check['BF_Ultimate_Model']).abs()
bf_check


,LatestReported,SelectedCDF,ExpectedUltimatePrior,BF_Ultimate_Manual,BF_Ultimate_Model,AbsDiff_Manual_vs_Model
AccidentYear,,,,,,
1998,NaN,1.000000,4.973157e+07,4.774230e+07,4.774230e+07,0.0
1999,NaN,1.000369,5.295079e+07,5.120532e+07,5.120532e+07,0.0
2000,NaN,1.000949,5.634034e+07,5.489133e+07,5.489133e+07,0.0
2001,NaN,1.002548,5.744853e+07,5.644559e+07,5.644559e+07,0.0
2002,NaN,1.005164,5.938451e+07,5.889778e+07,5.889778e+07,0.0
2003,NaN,1.009550,5.795169e+07,5.811356e+07,5.811356e+07,0.0
2004,NaN,1.020696,5.697666e+07,5.813196e+07,5.813196e+07,0.0
2005,NaN,1.048436,5.641034e+07,5.939249e+07,5.939249e+07,0.0
2006,NaN,1.109490,5.392237e+07,5.996266e+07,5.996266e+07,0.0


## Benktander Convergence Toward Chain Ladder

Benktander with `n_iters=1` is BF-like.
As `n_iters` increases, the estimate puts more weight on development and converges to Chain Ladder.


In [14]:
cl_model = cl.Chainladder().fit(selected_dev)
cl_total_ultimate = float(cl_model.ultimate_.sum())

benktander_rows = []
for n_iters in [1, 2, 3, 5, 10, 20, 50]:
    ben_model = cl.Benktander(apriori=selected_apriori, n_iters=n_iters).fit(selected_dev, sample_weight=sample_weight)
    ben_total_ultimate = float(ben_model.ultimate_.sum())
    ben_total_ibnr = float(ben_model.ibnr_.sum())
    abs_diff = abs(cl_total_ultimate - ben_total_ultimate)
    benktander_rows.append(
        {
            'n_iters': n_iters,
            'Benktander_TotalUltimate': ben_total_ultimate,
            'Benktander_TotalIBNR': ben_total_ibnr,
            'ChainLadder_TotalUltimate': cl_total_ultimate,
            'AbsDiff_to_CL': abs_diff,
            'PctDiff_to_CL': abs_diff / cl_total_ultimate,
        }
    )

benktander_convergence = pd.DataFrame(benktander_rows).set_index('n_iters')
benktander_convergence


,Benktander_TotalUltimate,Benktander_TotalIBNR,ChainLadder_TotalUltimate,AbsDiff_to_CL,PctDiff_to_CL
n_iters,,,,,
1,5.647962e+08,2.131462e+07,5.693014e+08,4.505234e+06,7.913617e-03
2,5.683713e+08,2.488975e+07,5.693014e+08,9.301041e+05,1.633764e-03
3,5.690948e+08,2.561325e+07,5.693014e+08,2.065994e+05,3.628999e-04
5,5.692905e+08,2.580894e+07,5.693014e+08,1.091247e+04,1.916817e-05
10,5.693014e+08,2.581984e+07,5.693014e+08,7.450760e+00,1.308755e-08
20,5.693014e+08,2.581985e+07,5.693014e+08,3.576279e-06,6.281872e-15
50,5.693014e+08,2.581985e+07,5.693014e+08,1.192093e-07,2.093957e-16


## AY-Level View: BF, Benktander, and Chain Ladder

This table compares AY ultimates for:
- BF (`n_iters=1` equivalent conceptually),
- Benktander at multiple iterations,
- Chain Ladder target.


In [15]:
ben_n1 = cl.Benktander(apriori=selected_apriori, n_iters=1).fit(selected_dev, sample_weight=sample_weight)
ben_n3 = cl.Benktander(apriori=selected_apriori, n_iters=3).fit(selected_dev, sample_weight=sample_weight)
ben_n10 = cl.Benktander(apriori=selected_apriori, n_iters=10).fit(selected_dev, sample_weight=sample_weight)

ay_compare = pd.DataFrame(
    {
        'BF_Ultimate': bf_model.ultimate_.to_frame().iloc[:, 0].values,
        'Benktander_n1': ben_n1.ultimate_.to_frame().iloc[:, 0].values,
        'Benktander_n3': ben_n3.ultimate_.to_frame().iloc[:, 0].values,
        'Benktander_n10': ben_n10.ultimate_.to_frame().iloc[:, 0].values,
        'ChainLadder_Ultimate': cl_model.ultimate_.to_frame().iloc[:, 0].values,
    },
    index=ay_year,
)
ay_compare['n1_to_CL_gap'] = ay_compare['ChainLadder_Ultimate'] - ay_compare['Benktander_n1']
ay_compare['n10_to_CL_gap'] = ay_compare['ChainLadder_Ultimate'] - ay_compare['Benktander_n10']
ay_compare.loc['Total'] = ay_compare.sum()
ay_compare


,BF_Ultimate,Benktander_n1,Benktander_n3,Benktander_n10,ChainLadder_Ultimate,n1_to_CL_gap,n10_to_CL_gap
AccidentYear,,,,,,,
1998,4.774230e+07,4.774230e+07,4.774230e+07,4.774230e+07,4.774230e+07,0.000000e+00,0.000000e+00
1999,5.120532e+07,5.120532e+07,5.120467e+07,5.120467e+07,5.120467e+07,-6.446515e+02,-7.450581e-09
2000,5.489133e+07,5.489133e+07,5.488995e+07,5.488995e+07,5.488995e+07,-1.374651e+03,0.000000e+00
2001,5.644559e+07,5.644559e+07,5.644303e+07,5.644303e+07,5.644303e+07,-2.555864e+03,7.450581e-09
2002,5.889778e+07,5.889778e+07,5.889527e+07,5.889527e+07,5.889527e+07,-2.513299e+03,-1.490116e-08
2003,5.811356e+07,5.811356e+07,5.811511e+07,5.811511e+07,5.811511e+07,1.545962e+03,-7.450581e-09
2004,5.813196e+07,5.813196e+07,5.815586e+07,5.815587e+07,5.815587e+07,2.391075e+04,-2.235174e-08
2005,5.939249e+07,5.939249e+07,5.953662e+07,5.953693e+07,5.953693e+07,1.444439e+05,1.415610e-07
2006,5.996266e+07,5.996266e+07,6.061757e+07,6.062401e+07,6.062401e+07,6.613492e+05,5.870759e-04


## Cape Cod Method: ECR Derivation and Setup

Cape Cod is closely related to BF, but it derives the expected claim ratio from reported data and used-up premium.

Formula view:
- Cape Cod ECR = Total Reported to Date / Total Used-up Premium
- Total Used-up Premium = sum(On-level Earned Premium x % Reported)

Educational notes:
- Only reported claims are used in the calibration year set.
- Premiums should be adjusted to a consistent on-level basis.


In [16]:
# On-level premium assumption by AY (example educational setup)
onlevel_factor_by_ay = pd.Series(
    [1.080, 1.070, 1.060, 1.050, 1.040, 1.030, 1.020, 1.010, 1.005, 1.000],
    index=ay_year,
    dtype=float,
)

onlevel_earned_premium = earned_premium_proxy * onlevel_factor_by_ay.values
percent_reported_by_ay = 1.0 / selected_cdf_by_ay.values
used_up_premium = onlevel_earned_premium * percent_reported_by_ay
cape_cod_ecr_manual = float(latest_reported.sum() / used_up_premium.sum())

cape_cod_weight = build_exposure_triangle(
    pd.Series(onlevel_earned_premium.values, index=ay_year, dtype=float),
    selected_dev,
)
cape_cod_model = cl.CapeCod(trend=0.0, decay=1.0, n_iters=1).fit(
    selected_dev,
    sample_weight=cape_cod_weight,
)
cape_cod_ecr_model = float(cape_cod_model.apriori_.to_frame().iloc[0, 0])

cape_cod_setup = pd.DataFrame(
    {
        'LatestReported': latest_reported.values,
        'SelectedCDF': selected_cdf_by_ay.values,
        'PctReported': percent_reported_by_ay,
        'OnLevelFactor': onlevel_factor_by_ay.values,
        'OnLevelEarnedPremium': onlevel_earned_premium.values,
        'UsedUpPremium': used_up_premium,
    },
    index=ay_year,
)
cape_cod_setup.index.name = 'AccidentYear'

pd.Series({
    'CapeCodECR_Manual': cape_cod_ecr_manual,
    'CapeCodECR_ModelApriori': cape_cod_ecr_model,
    'AbsDiff_Manual_vs_Model': abs(cape_cod_ecr_manual - cape_cod_ecr_model),
}), cape_cod_setup


(CapeCodECR_Manual          0.74435
 CapeCodECR_ModelApriori    0.74435
 AbsDiff_Manual_vs_Model    0.00000
 dtype: float64,
               LatestReported  SelectedCDF  PctReported  OnLevelFactor  \
 AccidentYear                                                            
 1998              47742304.0     1.000000     1.000000          1.080   
 1999              51185767.0     1.000369     0.999631          1.070   
 2000              54837929.0     1.000949     0.999052          1.060   
 2001              56299562.0     1.002548     0.997458          1.050   
 2002              58592712.0     1.005164     0.994863          1.040   
 2003              57565344.0     1.009550     0.990540          1.030   
 2004              56976657.0     1.020696     0.979723          1.020   
 2005              56786410.0     1.048436     0.953801          1.010   
 2006              54641339.0     1.109490     0.901315          1.005   
 2007              48853563.0     1.303780     0.767001      

## Cape Cod Results, Comparison, and Impacts

Advantages/disadvantages (formula-sheet style):
- Advantage: compared with pure development, Cape Cod can be more resilient to random AY volatility.
- Disadvantage: requires sufficient credible reported claims and careful on-level premium treatment.

The comparison below shows where Cape Cod lands relative to BF, Benktander, and Chain Ladder in this dataset.


In [17]:
cape_cod_ultimate = cape_cod_model.ultimate_.to_frame().iloc[:, 0]
cape_cod_ibnr = cape_cod_model.ibnr_.to_frame().iloc[:, 0]

cape_cod_compare = pd.DataFrame(
    {
        'CapeCod_Ultimate': cape_cod_ultimate.values,
        'BF_Ultimate': bf_model.ultimate_.to_frame().iloc[:, 0].values,
        'Benktander_n3_Ultimate': ben_n3.ultimate_.to_frame().iloc[:, 0].values,
        'ChainLadder_Ultimate': cl_model.ultimate_.to_frame().iloc[:, 0].values,
    },
    index=ay_year,
)
cape_cod_compare['CapeCod_to_CL_gap'] = cape_cod_compare['ChainLadder_Ultimate'] - cape_cod_compare['CapeCod_Ultimate']
cape_cod_compare.loc['Total'] = cape_cod_compare.sum()

cape_cod_totals = pd.Series({
    'CapeCod_TotalUltimate': float(cape_cod_model.ultimate_.sum()),
    'CapeCod_TotalIBNR': float(cape_cod_model.ibnr_.sum()),
    'BF_TotalUltimate': float(bf_model.ultimate_.sum()),
    'Benktander_n3_TotalUltimate': float(ben_n3.ultimate_.sum()),
    'ChainLadder_TotalUltimate': float(cl_model.ultimate_.sum()),
})

cape_cod_impact_table = pd.DataFrame(
    [
        ['Increase in exposure', 'No material effect (as long as average accident date is held constant)'],
        ['Average accident date shifts forward', 'Underestimates ultimate claims by less than development method but more than BF method'],
        ['Increase claim ratios', 'Underestimates ultimate claims unless most recent data is reflected in Cape Cod ECR'],
        ['Speedup in claim settlement rate', 'No material effect'],
        ['Increase in case outstanding adequacy', 'Overestimates ultimate claims by less than development method but more than BF method'],
        ['Change in product mix', 'Accuracy is impacted when lines have different development patterns or ECRs'],
    ],
    columns=['Description', 'Impact on Cape Cod method'],
)

cape_cod_totals, cape_cod_compare, cape_cod_impact_table


(CapeCod_TotalUltimate          5.647511e+08
 CapeCod_TotalIBNR              2.126948e+07
 BF_TotalUltimate               5.647962e+08
 Benktander_n3_TotalUltimate    5.690948e+08
 ChainLadder_TotalUltimate      5.693014e+08
 dtype: float64,
               CapeCod_Ultimate   BF_Ultimate  Benktander_n3_Ultimate  \
 AccidentYear                                                           
 1998              4.774230e+07  4.774230e+07            4.774230e+07   
 1999              5.120653e+07  5.120532e+07            5.120467e+07   
 2000              5.489410e+07  5.489133e+07            5.488995e+07   
 2001              5.645174e+07  5.644559e+07            5.644303e+07   
 2002              5.890759e+07  5.889778e+07            5.889527e+07   
 2003              5.812576e+07  5.811356e+07            5.811511e+07   
 2004              5.814619e+07  5.813196e+07            5.815586e+07   
 2005              5.939872e+07  5.939249e+07            5.953662e+07   
 2006              5.994898e

## Assumptions, Advantages, and Environmental Impacts

Advantages often cited:
- Works well when data credibility is limited (relative to pure development).
- Dampens volatility from random early fluctuations and single-year anomalies.

Impact intuition (when ECR is calibrated from historical data + development context):
- BF/Benktander usually sit between pure development and pure expected-claims responses.


In [18]:
impact_table = pd.DataFrame(
    [
        ['Increase in exposure', 'No material effect (if average accident date is held constant)', 'No material effect (if average accident date is held constant)'],
        ['Average accident date shifts forward', 'Underestimates ultimate by less than development method but more than expected claims method', 'Underestimates ultimate by less than development method but more than expected claims method'],
        ['Increase claim ratios', 'Underestimates if recent increase is not reflected in ECR; typically less than expected claims method miss', 'Underestimates if recent increase is not reflected in ECR; typically less than expected claims method miss'],
        ['Speedup in claim settlement rate', 'Can overestimate ultimate by less than development method but more than expected claims method', 'No material effect'],
        ['Increase in case outstanding adequacy', 'No material effect', 'Can overestimate ultimate by less than development method but more than expected claims method'],
        ['Change in product mix', 'Accuracy changes when segments have different development patterns or ECRs', 'Accuracy changes when segments have different development patterns or ECRs'],
    ],
    columns=['Description', 'Paid impact', 'Reported impact'],
)

summary_totals = pd.Series({
    'CL_TotalUltimate': cl_total_ultimate,
    'BF_TotalUltimate': float(bf_model.ultimate_.sum()),
    'BF_TotalIBNR': float(bf_model.ibnr_.sum()),
    'Benktander_n1_TotalUltimate': float(ben_n1.ultimate_.sum()),
    'Benktander_n10_TotalUltimate': float(ben_n10.ultimate_.sum()),
})

impact_table, summary_totals


(                             Description  \
 0                   Increase in exposure   
 1   Average accident date shifts forward   
 2                  Increase claim ratios   
 3       Speedup in claim settlement rate   
 4  Increase in case outstanding adequacy   
 5                  Change in product mix   
 
                                          Paid impact  \
 0  No material effect (if average accident date i...   
 1  Underestimates ultimate by less than developme...   
 2  Underestimates if recent increase is not refle...   
 3  Can overestimate ultimate by less than develop...   
 4                                 No material effect   
 5  Accuracy changes when segments have different ...   
 
                                      Reported impact  
 0  No material effect (if average accident date i...  
 1  Underestimates ultimate by less than developme...  
 2  Underestimates if recent increase is not refle...  
 3                                 No material effect  
 4